<a href="https://colab.research.google.com/github/internshipinabook/nlp-internship-in-a-book/blob/main/notebooks/week6_word_vectors_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⏸️ **Pause and Predict**
> Before loading any embeddings: what do you expect to find if you search for the 5 nearest neighbours of the word 'cancel'? Write your predictions, then run the search and compare.

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/nlp-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 6 — Word Vectors (STARTER)
### The NLP Internship | LinguaAI Labs

**Static vs Contextual:**
- **Static** (Word2Vec, FastText) — one fixed vector per word, ignores context
- **Contextual** (BERT) — vector changes with surrounding words → Week 8

CLINC150 queries are short (~8 words). A single OOV domain-specific word can lose the entire intent signal.

In [ ]:
import sys, subprocess, os
for pkg in ["datasets","scikit-learn","matplotlib","seaborn","gensim"]:
    try: __import__(pkg.replace("-","_"))
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
from datasets import load_dataset
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
import warnings; warnings.filterwarnings("ignore")
import gensim.downloader as api
np.random.seed(42); os.makedirs("outputs", exist_ok=True)

DOMAIN_MAP = {
    "bill_balance":"billing","bill_pay":"billing","transfer":"billing",
    "pay_bill":"billing","min_payment":"billing","account_blocked":"billing",
    "interest_rate":"billing","routing":"billing","spending_history":"billing",
    "credit_score":"account","credit_limit":"account","damaged_card":"account",
    "freeze_account":"account","replacement_card":"account","pin_change":"account",
    "lost_stolen_card":"account","cancel_card":"account","card_declined":"account",
    "report_fraud":"account","expiration_date":"account",
    "travel_alert":"travel","flight_status":"travel","travel_suggestion":"travel",
    "book_hotel":"travel","book_flight":"travel","exchange_rate":"travel",
    "reminder":"utility","reminder_update":"utility","calendar":"utility",
    "schedule_meeting":"utility","make_call":"utility","text":"utility",
    "timer":"utility","alarm":"utility","tell_joke":"utility",
}
clinc = load_dataset("clinc_oos", "plus")
label_names = clinc["train"].features["intent"].names
def collapse(i): return DOMAIN_MAP.get(label_names[i], "other")
df_train = clinc["train"].to_pandas().rename(columns={"text":"text_clean"})
df_test  = clinc["test"].to_pandas().rename(columns={"text":"text_clean"})
df_train["category"] = df_train["intent"].apply(collapse)
df_test["category"]  = df_test["intent"].apply(collapse)
X_train=df_train["text_clean"].values; y_train=df_train["category"].values
X_test =df_test["text_clean"].values;  y_test =df_test["category"].values
tfidf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=True)),
    ("clf",   LogisticRegression(multi_class="multinomial", max_iter=1000,
                                  class_weight="balanced", random_state=42)),
])
tfidf_pipeline.fit(X_train, y_train)
y_pred_tfidf = tfidf_pipeline.predict(X_test)
tfidf_f1 = f1_score(y_test, y_pred_tfidf, average="weighted", zero_division=0)
print(f"CLINC150 {len(X_train):,} train | Baseline F1={tfidf_f1:.3f} ✅")

## Part 1 — Word Similarities

> ⏸️ **Predict** — rank by similarity: (freeze,block), (flight,hotel), (reminder,alarm), (transfer,routing)

In [ ]:
word_vectors = api.load("word2vec-google-news-300")
print(f"Vocab: {len(word_vectors):,} | Dims: {word_vectors.vector_size}")
pairs = [("freeze","block"),("flight","hotel"),("reminder","alarm"),
         ("transfer","routing"),("credit","debit")]
for w1, w2 in pairs:
    try: print(f"  {w1} ↔ {w2}: {word_vectors.similarity(w1,w2):.3f}")
    except KeyError as e: print(f"  {e}: OOV")
print("\nCLINC terms in Word2Vec?")
for w in ["oos","routing_number","topup","freeze_account"]:
    print(f"  '{w}': {'✅' if w in word_vectors else '❌ OOV'}")

## Part 2 — Sentence Embeddings from Word2Vec

> 🧠 **Predict:** Query = 'What is my credit card routing number?' — how many tokens contribute to the average vector?

In [ ]:
def text_to_vector(text, wv, dim=300):
    """Average word vectors. Return zero vector if all OOV."""
    tokens = text.lower().split()
    # YOUR CODE HERE — collect wv[t] for each t in wv, then np.mean
    pass

X_train_w2v = np.vstack([text_to_vector(t, word_vectors) for t in X_train])
X_test_w2v  = np.vstack([text_to_vector(t, word_vectors) for t in X_test])
from sklearn.linear_model import LogisticRegression
clf_w2v = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
clf_w2v.fit(X_train_w2v, y_train)
y_pred_w2v = clf_w2v.predict(X_test_w2v)
w2v_f1 = f1_score(y_test, y_pred_w2v, average="weighted", zero_division=0)
print(f"Word2Vec + LR: {w2v_f1:.3f}  (TF-IDF: {tfidf_f1:.3f})")

## Part 3 — FastText (Handles OOV via Subwords)

In [ ]:
try: ft_vectors = api.load("fasttext-wiki-news-subwords-300")
except: ft_vectors = word_vectors; print("Using Word2Vec fallback")

def text_to_fasttext_vector(text, ft, dim=300):
    tokens = text.lower().split()
    # YOUR CODE HERE
    pass

X_train_ft = np.vstack([text_to_fasttext_vector(t, ft_vectors) for t in X_train])
X_test_ft  = np.vstack([text_to_fasttext_vector(t, ft_vectors) for t in X_test])
clf_ft = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
clf_ft.fit(X_train_ft, y_train)
y_pred_ft = clf_ft.predict(X_test_ft)
ft_f1 = f1_score(y_test, y_pred_ft, average="weighted", zero_division=0)
print(f"FastText + LR: {ft_f1:.3f}")

## Part 4 — Comparison

In [ ]:
print("=== CLINC150 — Representation Comparison ===\n")
for name_, f1, y_pred in [("TF-IDF+LR",tfidf_f1,y_pred_tfidf),
                            ("Word2Vec+LR",w2v_f1,y_pred_w2v),
                            ("FastText+LR",ft_f1,y_pred_ft)]:
    rep = classification_report(y_test,y_pred,output_dict=True,zero_division=0)
    acc_r = rep.get("account",{}).get("recall",0)
    print(f"  {name_:<14} F1={f1:.3f}  account_recall={acc_r:.2f}")
# 📚 Mikolov et al. (2013) arXiv:1301.3781 — Word2Vec

## 🏆 Challenge Task

> Train FastText on CLINC150 queries:
> ```python
> from gensim.models import FastText
> ft_custom = FastText(sentences=[t.split() for t in X_train], vector_size=100, min_count=1, epochs=15)
> ```
> Compare neighbours of 'freeze' and 'routing' vs pre-trained. Does domain training help?

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Using pre-trained embeddings without checking domain fit | GloVe was trained on Wikipedia and news. Your support tickets may use domain-specific terms that have no meaningful embedding. | Always check coverage: what fraction of your vocabulary is in the embedding? |
| Averaging word vectors for a document | Averaging loses word order and emphasis. 'I am not happy' and 'I am happy' have similar averages but opposite meanings. | For sentiment, consider weighted averaging by TF-IDF weight or use sentence transformers instead |

## ✅ What You Can Do After This Week

- Explain static vs contextual embeddings
- Build sentence embeddings by averaging word vectors
- Use FastText for OOV-heavy text via character n-grams
- Compare TF-IDF, Word2Vec, FastText on real intent data

---
## ✅ Week 6 Complete — Next: `week7_debugging_STARTER.ipynb`

---
*Add `week6_word_vectors_STARTER.ipynb` to your portfolio.*

*The NLP Internship · LinguaAI Labs · github.com/InternshipInABook*

## ✅ By completing Week 6, you can now:

- Explain what a word vector represents and why it captures meaning
- Check embedding coverage for your domain vocabulary
- Represent a document as an averaged word vector and explain the limitation
- Compare embedding-based and frequency-based representations on a real task

📤 **GitHub:** Push week6_word_vectors_STARTER.ipynb. Commit: "Week 6: Word vectors — GloVe coverage and similarity explored"


---

## 📚 Get the Full Book

This notebook is part of **The NLP Internship** — Book 2 of the InternshipInABook™ Series.

The full book includes:
- Complete week-by-week narrative and explanations
- All STOP AND TRACE code walkthroughs
- Fairness briefs, model cards, and deployment guides
- Certificate of Completion

👉 **[Get the book on Selar →](https://selar.com/8440iqfr61)**

---
*InternshipInABook™ Series · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook)*
